In [7]:
%matplotlib notebook
import os
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.widgets import SpanSelector
import ipywidgets as widgets
from IPython.display import display, clear_output

# --- 1. CONFIGURACIÓN DE RUTAS ---
CARPETA_DATOS = "/Users/aron/githubRepo/fall_risk/fall-dataset/SisFall_procesado/SA01"
CARPETA_GUARDADO = "/Users/aron/githubRepo/fall_risk/fall-dataset/SisFall_tagueados/SA01"
os.makedirs(CARPETA_GUARDADO, exist_ok=True)

# --- 2. GESTOR DEL ESTADO (Memoria del programa) ---
class Etiquetador:
    def __init__(self):
        archivos_csv = sorted([f for f in os.listdir(CARPETA_DATOS) if f.endswith('.csv')])
        archivos_ya_etiquetados = set(os.listdir(CARPETA_GUARDADO))
        self.archivos_pendientes = [f for f in archivos_csv if f not in archivos_ya_etiquetados]
        
        self.archivo_actual = None
        self.df_actual = None
        self.selector = None

estado = Etiquetador()

# --- 3. INTERFAZ GRÁFICA (Botones) ---
btn_siguiente = widgets.Button(description="Guardar y Siguiente", button_style='success', icon='check')
btn_saltar = widgets.Button(description="Saltar archivo", button_style='warning', icon='step-forward')
out = widgets.Output()

# --- 4. MOTOR LÓGICO ---
def cargar_siguiente(b=None):
    with out:
        clear_output(wait=True)
        
        # Si venimos del botón "Guardar", guardamos el CSV que estaba en memoria
        if b is btn_siguiente and estado.df_actual is not None and estado.archivo_actual is not None:
            ruta_destino = os.path.join(CARPETA_GUARDADO, estado.archivo_actual)
            estado.df_actual.to_csv(ruta_destino, index=False)
            print(f"💾 Guardado con éxito: {estado.archivo_actual}")

        # Comprobar si ya terminamos
        if not estado.archivos_pendientes:
            print("🎉 ¡Felicidades! Has terminado de etiquetar todos los archivos de esta carpeta.")
            return

        # Cargar el siguiente archivo en la cola
        estado.archivo_actual = estado.archivos_pendientes.pop(0)
        ruta_origen = os.path.join(CARPETA_DATOS, estado.archivo_actual)
        
        # Leer datos y preparar la columna tag
        estado.df_actual = pd.read_csv(ruta_origen)
        estado.df_actual['tag'] = 0 # Inicializamos todo en 0 por defecto
            
        print(f"🔍 Analizando: {estado.archivo_actual} | Pendientes: {len(estado.archivos_pendientes)}")
        print("👉 Instrucciones: Sombrea el impacto con el mouse. Si te equivocas, vuelve a sombrear.")
        print("👉 Si es una actividad normal (ADL), no sombrees nada, solo presiona 'Guardar y Siguiente'.\n")

        # Configurar la gráfica
        fig, ax = plt.subplots(figsize=(12, 4))
        ax.plot(estado.df_actual.index, estado.df_actual['az'], color='#1f77b4', linewidth=1.5)
        ax.set_title(f"Aceleración Z - {estado.archivo_actual}", fontweight='bold')
        ax.set_xlabel("Muestras (Tiempo)")
        ax.set_ylabel("Aceleración")
        ax.grid(True, linestyle='--', alpha=0.6)

        # La función que inyecta los '1' al sombrear
        def al_seleccionar(xmin, xmax):
            fila_inicio, fila_fin = int(xmin), int(xmax)
            estado.df_actual['tag'] = 0 # Borra selecciones anteriores si te equivocaste
            estado.df_actual.loc[fila_inicio:fila_fin, 'tag'] = 1
            print(f"✅ ¡Impacto capturado! Filas {fila_inicio} a {fila_fin} marcadas con '1'.")

        # Activar la herramienta del mouse
        estado.selector = SpanSelector(
            ax, al_seleccionar, direction='horizontal', useblit=True,
            props=dict(alpha=0.3, facecolor='red')
        )
        
        plt.show()

# --- 5. CONECTAR Y ARRANCAR ---
btn_siguiente.on_click(cargar_siguiente)
btn_saltar.on_click(lambda b: cargar_siguiente(btn_saltar)) # Le pasamos el botón saltar para que no guarde

# Mostrar los botones y la consola de salida
display(widgets.HBox([btn_siguiente, btn_saltar]), out)

# Disparar el primer archivo automáticamente
cargar_siguiente()

AttributeError: partially initialized module 'pandas' from '/Users/aron/githubRepo/fall_risk/python_code/clean/.venv/lib/python3.14/site-packages/pandas/__init__.py' has no attribute '_pandas_datetime_CAPI' (most likely due to a circular import)